In [1]:
#Create the grid CSV for the solubility calculations 200 solvents
import pandas as pd
import numpy as np

# 2. Load your dataset
file_path = 'Solvent List for calc.xlsx' 
df = pd.read_excel(file_path)

# .iloc[1:200] means start at row 1 and stop before row 200
df_subset = df[['Solvent', 'Smiles', 'MWt (g/mol)']].iloc[1:200].copy()

# 4. Define the temperature range (278.15 to 333.15 with delta T = 5)
temps = np.arange(278.15, 333.15 + 0.001, 5) 

# 5. Expand the dataset for each temperature
df_expanded = pd.DataFrame()

for temp in temps:
    temp_df = df_subset.copy()
    temp_df['Temperature'] = temp
    df_expanded = pd.concat([df_expanded, temp_df], ignore_index=True)

# 6. Sort by Solvent, then by Temperature
df_expanded = df_expanded.sort_values(by=['Solvent', 'Temperature']).reset_index(drop=True)

# 7. Add an empty 'LogS' column placeholder
df_expanded['LogS'] = np.nan

# 8. Save the new dataset with a different filename
output_filename = 'clozapine_pretrain_first_200.csv'
df_expanded.to_csv(output_filename, index=False)

print(f"Done! Created {len(df_expanded)} data points (Second 200 solvents × 12 temperatures).")
print(f"File saved as: {output_filename}")
print(df_expanded.head(10))

Done! Created 2388 data points (Second 200 solvents × 12 temperatures).
File saved as: clozapine_pretrain_first_200.csv
                  Solvent          Smiles  MWt (g/mol)  Temperature  LogS
0  (1-Thiapropyl)-Benzene  C1=CC=CC=C1SCC        138.2       278.15   NaN
1  (1-Thiapropyl)-Benzene  C1=CC=CC=C1SCC        138.2       283.15   NaN
2  (1-Thiapropyl)-Benzene  C1=CC=CC=C1SCC        138.2       288.15   NaN
3  (1-Thiapropyl)-Benzene  C1=CC=CC=C1SCC        138.2       293.15   NaN
4  (1-Thiapropyl)-Benzene  C1=CC=CC=C1SCC        138.2       298.15   NaN
5  (1-Thiapropyl)-Benzene  C1=CC=CC=C1SCC        138.2       303.15   NaN
6  (1-Thiapropyl)-Benzene  C1=CC=CC=C1SCC        138.2       308.15   NaN
7  (1-Thiapropyl)-Benzene  C1=CC=CC=C1SCC        138.2       313.15   NaN
8  (1-Thiapropyl)-Benzene  C1=CC=CC=C1SCC        138.2       318.15   NaN
9  (1-Thiapropyl)-Benzene  C1=CC=CC=C1SCC        138.2       323.15   NaN


In [ ]:
# generate_cosmo_files.py
# Env: needs rdkit, opi (ORCA python interface), ORCA on PATH
# Run inside your OPI venv

from pathlib import Path
import re
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem

from opi.core import Calculator
from opi.input.structures.structure import Structure

# ---- config ----
GRID_CSV = "clozapine_pretrain_first_200.csv"        # columns: Solvent, Smiles, Temperature
CLOZAPINE_SMILES = "ClC1=CC=C2NC=3C=CC=CC3C(=NC2=C1)N4CCN(C)CC4"  # exact SMILES from Clozapine_dataset.csv
WORKDIR = Path("cosmo_calcs")
NCORES = 4
WORKDIR.mkdir(exist_ok=True)

def safe_name(name: str) -> str:
    """Turn a solvent name into a filesystem-safe folder/basename."""
    return re.sub(r"[^A-Za-z0-9_-]+", "_", str(name)).strip("_")

def smiles_to_xyz_block(smiles: str, n_confs: int = 5, seed: int = 42) -> str:
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    cids = AllChem.EmbedMultipleConfs(mol, numConfs=n_confs, randomSeed=seed)
    energies = []
    for cid in cids:
        ff = AllChem.MMFFGetMoleculeForceField(mol, AllChem.MMFFGetMoleculeProperties(mol), confId=cid)
        if ff is None:
            ff = AllChem.UFFGetMoleculeForceField(mol, confId=cid)
        ff.Minimize(maxIts=2000)
        energies.append((ff.CalcEnergy(), cid))
    best_cid = min(energies, key=lambda t: t[0])[1]
    return Chem.MolToXYZBlock(mol, confId=best_cid)

def setup_calc(basename: str, working_dir: Path, structure: Structure, ncores: int = NCORES) -> Calculator:
    calc = Calculator(basename=basename, working_dir=working_dir)
    calc.structure = structure
    calc.input.add_arbitrary_string("!COSMORS(water)")
    calc.input.ncores = ncores
    return calc

def run_and_get_cosmo(smiles: str, name: str) -> Path | None:
    mol_dir = WORKDIR / name
    mol_dir.mkdir(exist_ok=True)
    out_file = mol_dir / f"{name}.solute.orcacosmo"
    if out_file.exists():
        return out_file 

    try:
        xyz = smiles_to_xyz_block(smiles)
    except Exception as e:
        print(f"[{name}] RDKit embedding failed: {e}")
        return None

    xyz_path = mol_dir / f"{name}.xyz"
    xyz_path.write_text(xyz)
    structure = Structure.from_xyz(xyz_path)

    calc = setup_calc(basename=name, working_dir=mol_dir, structure=structure)
    calc.write_input()
    try:
        calc.run()
    except Exception as e:
        print(f"[{name}] ORCA run failed: {e}")
        return None

    output = calc.get_output()
    if not output.terminated_normally():
        print(f"[{name}] ORCA did not terminate normally")
        return None

    return out_file if out_file.exists() else None

def main():
    grid_df = pd.read_csv(GRID_CSV)
    unique_solvents = grid_df.drop_duplicates(subset="Smiles")[["Solvent", "Smiles"]].reset_index(drop=True)
    print(f"{len(grid_df)} grid rows -> {len(unique_solvents)} unique solvents to run through ORCA")

    log_rows = []

    # --- solute: clozapine, done once ---
    cosmo_path = run_and_get_cosmo(CLOZAPINE_SMILES, "clozapine")
    log_rows.append({"name": "clozapine", "smiles": CLOZAPINE_SMILES, "cosmo_file": str(cosmo_path)})

    # --- solvents ---
    for i, row in unique_solvents.iterrows():
        smi = row["Smiles"]
        name = safe_name(row["Solvent"])
        print(f"[{i+1}/{len(unique_solvents)}] {name}: {smi}")
        cosmo_path = run_and_get_cosmo(smi, name)
        log_rows.append({"name": name, "solvent": row["Solvent"], "smiles": smi, "cosmo_file": str(cosmo_path)})

    pd.DataFrame(log_rows).to_csv(WORKDIR / "cosmo_manifest.csv", index=False)
    print("Done. Manifest written to", WORKDIR / "cosmo_manifest.csv")

if __name__ == "__main__":
    main()

2400 grid rows -> 200 unique solvents to run through ORCA
[1/200] 1-Thiapropyl_-Benzene: C1=CC=CC=C1SCC
[2/200] E_-Citral: CC(=CCC/C(=C/C=O)/C)C
[3/200] Z_-3-Hexen-1-Yl_Formate: CC\C=C/CCOC=O
[4/200] Z_-3-Hexen-1-ol: CC\C=C/CCO
[5/200] Z_-4-Hepten-1-ol: CC\C=C/CCCO
[6/200] Z_-6-Nonen-1-Al: CC\C=C/CCCCC=O
[7/200] Z_-Verbenol: CC1=C[C@H]([C@@H]2C[C@H]1C2(C)C)O
[8/200] 1_1-Dimethylcyclohexane: CC1(C)CCCCC1
[9/200] 1_2_3-Trimethylbenzene: CC(C(C)=CC=C1)=C1C
[10/200] 1_2-Cyclohexanedione: C1CCC(=O)C(=O)C1
[11/200] 1_3-Butanediol: CC(O)CCO
[12/200] 1_3-Cyclohexadiene: C1=CCCC=C1
[13/200] 1_3-Dioxolane: C1OCCO1
[14/200] 1_4-Thioxane: C1CSCCO1
[15/200] 1_8-Cineole_Eucalyptol: CC1(C2CCC(O1)(CC2)C)C
[16/200] 1-Hepten-3-ol: CCCCC(C=C)O
[17/200] 1-Nonanal: CCCCCCCCC=O
[18/200] 1-Nonyne: C#CCCCCCCC
[19/200] 1-Octanal: CCCCCCCC=O
[20/200] 1-Octanol: CCCCCCCCO
[21/200] 1-Penten-3-ol: CCC(C=C)O
[22/200] 1-Phenyl-3-Methylbutane: CC(C)CCC1=CC=CC=C1
[23/200] 1-Propene_3_3_-Thiobis-: C=CCSCC=C
[24/200] 2_

In [ ]:
# compute_solubility_grid.py UPDATED VERSION
import numpy as np
import pandas as pd
from scipy.optimize import brentq

from opencosmorspy.parameterization import openCOSMORS24a
from opencosmorspy.cosmors import COSMORS

# ---- config ----
GRID_CSV = "clozapine_pretrain_first_200.csv"              # columns: Solvent, Smiles, MWt (g/mol), Temperature, LogS (empty)
MANIFEST_CSV = "cosmo_calcs/cosmo_manifest.csv"
OUT_CSV = "clozapine_pure_solvent_pretrain.csv"

R = 8.314
DELTA_H_FUSION = 34350.0
T_FUSION = 456.43
MW_CLOZAPINE = 326.82

# wide log10(x) scan range used to locate the sign change before bracketing
SCAN_LOG10_LO = -20.0
SCAN_LOG10_HI = np.log10(0.3)
SCAN_POINTS = 40

def compute_ln_gamma(crs, mole_fractions, temp):
    crs.clear_jobs()
    crs.add_job(np.array(mole_fractions), temp, refst="pure_component")
    return crs.calculate()["tot"]["lng"][0][0]

def solubility_non_iterative(crs, delta_h_fus, t_fus, temp):
    """Fast closed-form estimate using ln(gamma) at infinite dilution.
    Assumes gamma stays roughly constant near x -> 0; used as a
    sanity check alongside the iterative (brentq) result."""
    rhs = -delta_h_fus / R * (1 / temp - 1 / t_fus)
    ln_gamma_inf = compute_ln_gamma(crs, [0.0, 1.0], temp)
    return float(np.exp(rhs - ln_gamma_inf))

def residual_fn(crs, temp, rhs):
    def residual(log10_x):
        x = 10 ** log10_x
        x = min(max(x, 1e-300), 1 - 1e-15)
        mole_fracs = np.array([x, 1 - x])
        ln_gamma = compute_ln_gamma(crs, mole_fracs, temp)
        return ln_gamma + np.log(x) - rhs
    return residual

def find_bracket(residual, lo=SCAN_LOG10_LO, hi=SCAN_LOG10_HI, n=SCAN_POINTS):
    """Scan log10(x) space to find a sign change, return (lo, hi) bracket for brentq."""
    grid = np.linspace(lo, hi, n)
    vals = [residual(g) for g in grid]
    for i in range(len(grid) - 1):
        if vals[i] == 0:
            return grid[i], grid[i]
        if vals[i] * vals[i + 1] < 0:
            return grid[i], grid[i + 1]
    return None  # no sign change found anywhere in the scan range

def solve_solubility_x(crs, delta_h_fus, t_fus, temp):
    """Solve for solubility mole fraction using van't Hoff + COSMO-RS,
    with a bounded, monotonic solver (brentq) instead of unconstrained fsolve."""
    rhs = -delta_h_fus / R * (1 / temp - 1 / t_fus)
    residual = residual_fn(crs, temp, rhs)

    bracket = find_bracket(residual)
    if bracket is None:
        raise RuntimeError(
            f"No sign change found for T={temp}K across log10(x) in "
            f"[{SCAN_LOG10_LO}, {SCAN_LOG10_HI:.2f}] — gamma may not be behaving as expected"
        )

    lo, hi = bracket
    log10_x_sol = brentq(residual, lo, hi, xtol=1e-12)
    return float(10 ** log10_x_sol)

def mole_frac_to_logS(x_solute: float, mw_solute: float, mw_solvent: float) -> float:
    mass_solute = x_solute * mw_solute
    mass_solvent = (1 - x_solute) * mw_solvent
    w = mass_solute / (mass_solute + mass_solvent)
    return np.log10(100 * w)

def main():
    grid_df = pd.read_csv(GRID_CSV)
    manifest = pd.read_csv(MANIFEST_CSV)

    clozapine_row = manifest[manifest["name"] == "clozapine"].iloc[0]
    solute_path = clozapine_row["cosmo_file"]

    solvent_manifest = manifest[manifest["name"] != "clozapine"][["smiles", "cosmo_file"]]
    merged = grid_df.merge(solvent_manifest, left_on="Smiles", right_on="smiles", how="left")

    missing_cosmo = merged["cosmo_file"].isna().sum()
    if missing_cosmo:
        print(f"[warning] {missing_cosmo} rows have no matching cosmo_file — these will be skipped")

    crs = COSMORS(par=openCOSMORS24a())
    results = []
    current_solvent_smi = None

    merged = merged.sort_values("Solvent").reset_index(drop=True)

    for i, row in merged.iterrows():
        smi = row["Smiles"]
        T = row["Temperature"]
        cosmo_file = row["cosmo_file"]
        mw_solvent = row["MWt (g/mol)"]

        if pd.isna(cosmo_file) or pd.isna(mw_solvent):
            continue

        if smi != current_solvent_smi:
            crs.clear_molecules()
            crs.add_molecule([solute_path])
            crs.add_molecule([cosmo_file])
            current_solvent_smi = smi

        try:
            x_non_iter = solubility_non_iterative(crs, DELTA_H_FUSION, T_FUSION, T)
            x_sol = solve_solubility_x(crs, DELTA_H_FUSION, T_FUSION, T)
            logS_non_iter = mole_frac_to_logS(x_non_iter, MW_CLOZAPINE, float(mw_solvent))
            logS = mole_frac_to_logS(x_sol, MW_CLOZAPINE, float(mw_solvent))
        except Exception as e:
            print(f"[{row['Solvent']}, {T}K] solve failed: {e}")
            continue

        results.append({
            "Solvent": row["Solvent"],
            "SMILES_1": smi,
            "comp_1": 100.0,
            "Temperature": T,
            "MWt (g/mol)": mw_solvent,
            "x_solute_non_iter": x_non_iter,
            "x_solute": x_sol,
            "LogS_non_iter": logS_non_iter,
            "LogS": logS,
        })

        if (i + 1) % 50 == 0:
            print(f"[{i+1}/{len(merged)}] rows processed")

    out_df = pd.DataFrame(results)
    out_df.to_csv(OUT_CSV, index=False)
    print(f"Wrote {len(out_df)} rows to {OUT_CSV}")

if __name__ == "__main__":
    main()

[50/2400] rows processed
[100/2400] rows processed
[150/2400] rows processed
[200/2400] rows processed
[250/2400] rows processed
[300/2400] rows processed
[350/2400] rows processed
[400/2400] rows processed
[450/2400] rows processed
[500/2400] rows processed
[550/2400] rows processed
[600/2400] rows processed
[650/2400] rows processed
[700/2400] rows processed
[750/2400] rows processed
[800/2400] rows processed
[850/2400] rows processed
[900/2400] rows processed
[950/2400] rows processed
[1000/2400] rows processed
[1050/2400] rows processed
[1100/2400] rows processed
[1150/2400] rows processed
[1200/2400] rows processed
[1250/2400] rows processed
[1300/2400] rows processed
[1350/2400] rows processed
[1400/2400] rows processed
[1450/2400] rows processed
[1500/2400] rows processed
[1550/2400] rows processed
[1600/2400] rows processed
[1650/2400] rows processed
[1700/2400] rows processed
[1750/2400] rows processed
[1800/2400] rows processed
[1850/2400] rows processed
[1900/2400] rows proc